# <center> **Machine Learning for Social Data Science** <center/>

## <center> **Problem Set 2** <center/>

### *<center> Title (1,500 words) <center/>*

#### **Preprocessing**

In [ ]:
#Load in libraries
from docx import Document
import csv
import os
import pandas as pd

In [2]:
folder_path = "C:/Users/Ryan Whitehead/Desktop/OneDrive/Documents/Levelling Up/Funds/Community Ownership Fund/Project Profiles"

data = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".docx"):
            path = os.path.join(root, file)
            doc = Document(path)
            full_text = []
            filename_without_docx = os.path.splitext(file)[0]
            project_number, project_name = filename_without_docx.split(" – ", 1)
            for para in doc.paragraphs:
                if para.text.strip():
                    full_text.append(para.text.strip())
            text = " ".join(full_text)
            data.append({
                "Project_number": project_number,
                "Project_name": project_name,
                "Round": os.path.basename(root),
                "Text": text})

In [3]:
#Remove project number and – from text
for entry in data:
    project_prefix = f"{entry['Project_number']} – "
    if entry['Text'].startswith(project_prefix):
        entry['Text'] = entry['Text'][len(project_prefix):].strip()

In [4]:
#Regex
import re

# 1. Create a regex pattern to identify URLs starting with https://
# 'https://' matches the literal string.
# '\S+' matches one or more non-whitespace characters (the rest of the URL).
url_pattern = r'https://\S+'

# 2. Create a regex pattern to identify access dates
# '\[' and '\]' are used to match the literal square bracket characters.
# 'Date Accessed:' matches the literal text.
# '.*?' non-greedily matches any characters (the date itself) up until the closing bracket.
date_pattern = r'\[Date Accessed:.*?\]'

# 3. Write a loop to clean the raw text for each entry in the dataset
for entry in data:
    # Check if 'Text' exists in the dictionary and is a string type just to be safe
    if 'Text' in entry and isinstance(entry['Text'], str):
        
        # Extract the current raw text
        current_text = entry['Text']
        
        # Use re.sub() to substitute the URL pattern with an empty string ('')
        text_without_urls = re.sub(url_pattern, '', current_text)
        
        # Use re.sub() to substitute the Date pattern with an empty string ('')
        text_without_dates = re.sub(date_pattern, '', text_without_urls)
        
        # Strip removes any extra whitespace left at the start or end of the string
        cleaned_text = text_without_dates.strip()
        
        # 4. Update the dictionary with the fully cleaned text
        entry['Text'] = cleaned_text

# Optional: Print the first few entries to verify it worked
#print(data[:5])


In [5]:
#Convert to Pandas Df and save corpus as CSV file for future ease and use
df = pd.DataFrame(data)
df.to_csv("Levelling_Up_COF_Corpus.csv", index = False)

#### **Topic Modelling**

In [6]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [ ]:
vectoriser_model = CountVectorizer(stop_words = 'english', min_df = 2, ngram_range = (1, 2))
representation_model = KeyBERTInspired()

topic_model = BERTopic(
    vectorizer_model = vectoriser_model,
    representation_model = representation_model)

BERTopic(calculate_probabilities=False, ctfidf_model=ClassTfidfTransformer(...), embedding_model=None, hdbscan_model=HDBSCAN(...), language=english, low_memory=False, min_topic_size=10, n_gram_range=(1, 1), nr_topics=None, representation_model=KeyBERTInspired(...), seed_topic_list=None, top_n_words=10, umap_model=UMAP(...), vectorizer_model=CountVectorizer(...), verbose=False, zeroshot_min_similarity=0.7, zeroshot_topic_list=None)


#### **Validation**